# Composite Index
Greatly inspired by S&P500 model. Reason to create composite index based on the fact that RSI_SMA_DIST is white noise where mean lies at 0 and variance is heteroskedastic. Accept that it cannot be predicted. Instead, we want to create a composite index where mean has a certain bias (upside)
- RSI(14) - SMA(14) mean is either > 20 or < 20.
- Best result: Finding a mean such that when subtracted by variance, it still stays > 0.

Three possible approaches:
- Let assets with RSI(14) - SMA(14) > 0 run. Cut out assets with RSI(14) - SMA(14) < 0.
- Let assets that score strong trend in HMM run. Cut out assets that is no longer in a strong trend.
- Create multi-variate synthetic asset. Use kalman with frozen beta for stability or use LAsSO so it can default losers to 0 and let winners run.

In [1]:
# Add the project root (one level up from /research) to the path
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Libraries needed to run the code
from utils.bybit import BybitConnector
from utils.screener import *

client = BybitConnector(api_key="", api_secret="", testnet=False)

# Testing out Screener
Result shows that no asset has a strong bias to one side. Meaning, it confirms the fact that we need to create a synthetic asset. At minimum, bias_std2 - bias > 0 or < 0 which means assumption-wise 95% of the time asset remains on one side of the trend.

In [2]:
# 1. Get your raw data
screened_data = screener(client)

# 2. Get the ranked table
ranked_df = rank_bias(screened_data)

# 3. Print the Top 10 strongest trending assets
print("--- Top 10 Assets by Bias Intensity ---")
print(ranked_df.head(10))

# 4. Optional: Filter for specific directional bias
strong_bullish = ranked_df[ranked_df['bias'] > 0].head(5)
strong_bearish = ranked_df[ranked_df['bias'] < 0].head(5)

Total tickers: 2682 | After filtering: 680
Skipping BSP/USDT:USDT: Insufficient data (192)
Skipping NFLX/USDT:USDT: Insufficient data (201)
Error on RECALL/USDT:USDT: CCXT / Bybit API Error: bybit {"retCode":10006,"retMsg":"Too many visits. Exceeded the API Rate Limit.","result":{},"retExtInfo":{},"time":1783339205312}
Error on RESOLV/USDT:USDT: CCXT / Bybit API Error: bybit {"retCode":10006,"retMsg":"Too many visits. Exceeded the API Rate Limit.","result":{},"retExtInfo":{},"time":1783339218112}
Error on RLUSD/USDT:USDT: CCXT / Bybit API Error: bybit {"retCode":10006,"retMsg":"Too many visits. Exceeded the API Rate Limit.","result":{},"retExtInfo":{},"time":1783339227818}
Error on RLC/USDT:USDT: CCXT / Bybit API Error: bybit {"retCode":10006,"retMsg":"Too many visits. Exceeded the API Rate Limit.","result":{},"retExtInfo":{},"time":1783339228943}
Error on ROBO/USDT:USDT: CCXT / Bybit API Error: bybit {"retCode":10006,"retMsg":"Too many visits. Exceeded the API Rate Limit.","result":{}

# Composite Index

In [ ]:
# Wire this up to however you build your connector elsewhere in your
# codebase (same object screener.screener() expects).
from utils.screener import screener
from utils.composite import *
from utils.bybit import BybitConnector  # <-- adjust to your actual class name

connector = BybitConnector()
screened_data = screener(client)

composite_df, membership, active = build_composite(screened_data, period=7)
bias, rsi_sma_dist = test_composite_persistency(composite_df)

print(f"Candidate assets: {len(screened_data)}")
print(f"Current constituents: {int(active.iloc[-1].sum())}")
print(f"Composite's own 200-bar trend-persistency bias: {bias[-1]:.4f}")

changes = membership_changes(membership)
print(changes.tail(20))

Total tickers: 2690 | After filtering: 680
